In [38]:
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import os
import json
import asyncio
from dotenv import load_dotenv
x = load_dotenv()

from typesafe_sdk import Choice, TypeSafeClient, AsyncTypeSafeClient


In [39]:
async def model_list() -> None:
    async with AsyncTypeSafeClient() as client:
        return await client.models.list()

def format_duration(td):
    total_seconds = int(td.total_seconds())
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    return f"{hours:02}:{minutes:02}:{seconds:02} (h:m:s)"

def create_metadata(instructions, criteria, task, criteria_dict, instructions_dict):
    meta = {}
    meta['Title'] = f'Task{task}_ins-{instructions}_cri-{criteria}'
    meta['Date'] = datetime.now().strftime('%Y/%m/%d %H:%M')
    meta['Task'] = task

    meta['instructions_id'] = instructions
    meta['instructions'] = instructions_dict[instructions]
    meta['criteria_id'] = criteria
    meta['criteria'] = criteria_dict[criteria]

    dump = TypeSafeClient().models.list().model_dump()
    meta['model'] = 'jev-latest'
    meta['model_metadata'] = next((m for m in dump['models'] if m['name'] == meta['model']), None)

    path = f"results/{meta['Title']}.json"
    i = 0

    while os.path.exists(path):
        i += 1
        path = f"results/{meta['Title']}"
        path = f"{path}_{i}.json"

    with open(f'{path}', 'x', encoding='utf-8') as f:
        json.dump(meta, f, indent=4, ensure_ascii=False)
        print(f'Saved {path}')

    return path

def ask_jev(instructions, criteria, state):
    starttime = datetime.now()
    response = client.system_one(
        state=state,
        questions={
            "Task": Choice(
                instructions=instructions,
                criteria=criteria,
            )
        },
    )
    endtime = datetime.now()
    time = str(endtime - starttime)

    return response, time

async def async_jev(instructions, criteria, state):
    starttime = datetime.now()
    response = await client.system_one(
        state=state,
        questions={
            "Task": Choice(
                instructions=instructions,
                criteria=criteria,
            )
        },
    )
    endtime = datetime.now()
    time = str(endtime - starttime)

    return response, time


def print_summary(metadata):
    with open(metadata, 'r', encoding='utf-8') as js:
        meta = json.load(js)
    if os.path.exists(metadata.replace('.json', '.csv')):
        results_df = pd.read_csv(metadata.replace('.json', '.csv'))
    else:
        print('No results available yet.')
        return

    high_conf = results_df[results_df['confidence'].astype(float)>=0.8]
    mid_conf = results_df[(results_df['confidence'].astype(float)<0.8) & (results_df['confidence'].astype(float)>=0.5)]
    low_conf = results_df[results_df['confidence'].astype(float)<0.5]

    summary = f"""
        Summary of results for {meta['Title']}:
        Date: {meta['Date']}
        Tokens used: {int(meta['Input Tokens']) + int(meta['Output Tokens'])}
        Time taken: {meta['Duration']}

        Total: {results_df.shape[0]} results, {round(results_df['correct'].mean()*100, 2)}% correct.
        Confidence above 80%: {high_conf.shape[0]} results, {round(high_conf['correct'].mean()*100, 2)}% correct.
        Confidence between 50 and 80%: {mid_conf.shape[0]} results, {round(mid_conf['correct'].mean()*100, 2)}% correct.
        Confidence below 50%: {low_conf.shape[0]} results, {round(low_conf['correct'].mean()*100, 2)}% correct.\n
    """
    return summary

def save_data(metadata, results_df):
    with open(metadata, 'r', encoding='utf-8') as js:
        meta = json.load(js)
    results_df.to_csv(metadata.replace('.json', '.csv'), index=False)

    meta['Input Tokens'] = int(results_df['input_tokens'].astype(int).sum())
    meta['Output Tokens'] = int(results_df['output_tokens'].astype(int).sum())
    meta['Duration'] = format_duration(pd.to_timedelta(results_df['time']).sum())
    meta['Accuracy'] = f'{round(results_df['correct'].mean()*100, 2)}%'

    with open(f'{metadata}', 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=4, ensure_ascii=False)
        print(f'Saved {metadata}')

    print(print_summary(metadata))

    return metadata
    

In [32]:
criteria_dict = {
    'basic': {
        "contains": "",
        "within": "",
        "touches": "",
        "crosses": "",
        "disjoint": "",
        "overlaps": "",
        "equals": "",
    },
    'explained': {}
}

In [33]:
instructions_dict = {
    'basic': "What topological predicate best describes the relationship between Place A and Place B? The predicates follow the definition of DE-9IM.",
    'think': "What topological predicate best describes the relationship between Place A and Place B? The predicates follow the definition of DE-9IM. Think step by step.", # Thinking step by step is contradictory to system one intelligence
    'ji': "You will be given the WKT format of geometries given the subject A and reference object B. Classify their spatial relations into one of the predicates defined by DE-9IM and Open Geospatial Consortium. If A equals to B, there is no need to return 'within' or 'contains'."
}

## Task 1 — Topological spatial relation qualification

Reproduction of the experiment done by Ji et al (2025) using Jev by TypesafeAI instead of GPT-4. The same data and evaluation methods are used. Prompts are adapted to the Jev model and different prompting strategies (instructions and criteria) are tested.

Input: Two WKT geometries
Output: Topological predicate


Ji, Y., Gao, S., Nie, Y., Majić, I., & Janowicz, K. (2025). Foundation models for geospatial reasoning: Assessing the capabilities of large language models in understanding geometries and topological spatial relations. International Journal of Geographical Information Science, 39(9), 1866–1903. https://doi.org/10.1080/13658816.2025.2511227

In [ ]:
# SYNCHRONOUS EXECUTION
client = TypeSafeClient()

instructions = 'think'
criteria = 'basic'
path = create_metadata(instructions, criteria, '1', criteria_dict, instructions_dict)
results = []

df = pd.read_csv('relations.csv')
df = df[:1]

for row in tqdm(df.itertuples(), total=len(df), desc="Prompting"):
    result = {}
    state = f"Geometry A: {row.geometry}, Geometry B: {row.geometry_subject}"
    response, time = ask_jev(instructions_dict[instructions], criteria_dict[criteria], state)

    result['GeomA'] = row.geometry
    result['GeomB'] = row.geometry_subject 
    result['predicate'] = row.predicate
    result['choice'] = response.answers['Task'].choice
    result['probabilities'] = str(response.answers['Task'].probabilities)
    result['confidence'] = response.answers['Task'].confidence
    result['correct'] = int(result['choice'] == result['predicate'])
    result['input_tokens'] = response.usage.input_tokens
    result['output_tokens'] = response.usage.output_tokens
    result['time'] = time
    
    results.append(result)

results_df = pd.DataFrame(results)
save_data(path, results_df)

Saved results/Task1_ins-think_cri-basic_3.json


Processing geometries: 100%|██████████| 1/1 [00:20<00:00, 20.61s/it]

Saved results/Task1_ins-think_cri-basic_3.json

        Summary of results for Task1_ins-think_cri-basic:
        Date: 2026/10/19 17:34
        Tokens used: 506
        Time taken: 00:00:20 (h:m:s)

        Total: 1 results, 0.0% correct.
        Confidence above 80%: 1 results, 0.0% correct.
        Confidence between 50 and 80%: 0 results, nan% correct.
        Confidence below 50%: 0 results, nan% correct.

    


'results/Task1_ins-think_cri-basic_3.json'

In [46]:
# ASYNCHROUNOUS EXECUTION

from tqdm.asyncio import tqdm 
df = pd.read_csv('relations.csv')
df = df[:10]

client = AsyncTypeSafeClient()

instructions = 'think'
criteria = 'basic'
path = create_metadata(instructions, criteria, '1', criteria_dict, instructions_dict)

async def main():
    results = []
    tasks = []
    
    for row in df.itertuples():
        state = f"Geometry A: {row.geometry}, Geometry B: {row.geometry_subject}"
        task = async_jev(instructions_dict[instructions], criteria_dict[criteria], state)
        tasks.append(task)

    results_raw = await tqdm.gather(*tasks, desc="Prompting")

    for i, (response, time) in enumerate(results_raw):
        row = df.iloc[i] # Match the result to the original row
        result = {
            'GeomA': row.geometry,
            'GeomB': row.geometry_subject,
            'predicate': row.predicate,
            'choice': response.answers['Task'].choice,
            'probabilities': str(response.answers['Task'].probabilities),
            'confidence': response.answers['Task'].confidence,
            'correct': int(response.answers['Task'].choice == row.predicate),
            'input_tokens': response.usage.input_tokens,
            'output_tokens': response.usage.output_tokens,
            'time': time
        }
        results.append(result)

    results_df = pd.DataFrame(results)
    save_data(path, results_df)


await main()


Saved results/Task1_ins-think_cri-basic.json


Prompting: 100%|██████████| 10/10 [00:00<00:00, 14.50it/s]

Saved results/Task1_ins-think_cri-basic.json

        Summary of results for Task1_ins-think_cri-basic:
        Date: 2026/10/19 17:53
        Tokens used: 5624
        Time taken: 00:00:06 (h:m:s)

        Total: 10 results, 0.0% correct.
        Confidence above 80%: 1 results, 0.0% correct.
        Confidence between 50 and 80%: 3 results, 0.0% correct.
        Confidence below 50%: 6 results, 0.0% correct.

    
